# Liver cell identity in MERFISH

This notebook follows the biological workflow from real input data to a trained model, a newly selected panel, and manuscript-matched biological analyses. It does not read packaged aggregate results as tutorial inputs. [Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/agent_section/05_SMITH_Agent_Evaluation_source.ipynb).

## Biological question

Which genes should be measured in a spatial liver assay so that cell identities and their expression programs remain interpretable? The source snRNA-seq provides broad cell-state coverage, while spatial references add tissue context; the experiment compares a source-only panel with a panel informed by both kinds of biological evidence before reading out MERFISH.

**How to read the endpoint:** MERFISH cell-type accuracy asks whether the selected genes retain cellular identity in the assay that will be measured. Mean MERFISH expression asks whether the panel is supported by detectable biology. The paired comparison tests whether spatial references add biological information beyond the source transcriptome.

## Step 0: Download the real input data

Download the versioned Zenodo archive and verify its checksums before training:

```bash
python scripts/download_tutorial_data.py \
  --case 05_agent \
  --data-root data/tutorials
```

The notebook is pre-executed for documentation. Read the Docs does not download large data or train SMITH during documentation builds.

## Configuration

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
from IPython.display import Image, Markdown, display

def find_repository(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook inside a SMITH repository checkout.")

ROOT = find_repository(Path.cwd().resolve())
DATA_ROOT = Path(os.environ.get("SMITH_TUTORIAL_DATA", "data/tutorials")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("SMITH_TUTORIAL_OUTPUT", "outputs/tutorials")).expanduser().resolve()
CASE_OUTPUT = OUTPUT_ROOT / 'agent'
EPOCHS = int(os.environ.get("SMITH_TUTORIAL_EPOCHS", 30))
DEVICE = os.environ.get("SMITH_TUTORIAL_DEVICE", 'cpu')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


## Step 1: Inspect the biological input data

The inputs are healthy-liver snRNA-seq, MERFISH, and spatial-reference H5AD files. MERFISH is the held-out assay used for evaluation; the snRNA-seq and spatial references are training views that contribute complementary cell-state and tissue-location information.

In [ ]:
inputs = ['agent/liver_merfish/adata_healthy_nucseq.h5ad', 'agent/liver_merfish/adata_healthy_merfish.h5ad', 'agent/references/PSC011_C1_visium.h5ad', 'agent/references/WSSS_F_IMMsp9838712_visium.h5ad']
input_checksums = {}
for relative in inputs:
    path = DATA_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run scripts/download_tutorial_data.py first.")
    input_checksums[relative] = sha256_file(path)


## Step 2: Train SMITH and select a panel

SMITH is trained separately on the source and each spatial reference after restricting them to the MERFISH gene universe. Source and reference rankings are aggregated into source-only and multi-reference panels, which are newly written for each seed and panel size.

The command below starts from the H5AD inputs above and writes a fresh model ranking, panel, evaluation, and run manifest.

In [ ]:
command = [sys.executable, str(ROOT / 'reproducibility/workflows/agent/run_tutorial.py'), "--data-root", str(DATA_ROOT), "--output-dir", str(CASE_OUTPUT), "--device", DEVICE, "--epochs", str(EPOCHS)] + ['--reference', 'references/PSC011_C1_visium.h5ad', '--reference', 'references/WSSS_F_IMMsp9838712_visium.h5ad', '--panel-sizes', '32,64,128', '--training-seeds', '1,2', '--max-cells', '3000'] + ["--force"]
completed = subprocess.run(command, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if completed.returncode:
    print(completed.stdout)
    raise subprocess.CalledProcessError(completed.returncode, command)
manifest = json.loads((CASE_OUTPUT / "run_manifest.json").read_text())
if not manifest.get("training_runs"):
    raise RuntimeError("The workflow did not record any SMITH training runs.")
for relative in ['figure_data/figure6_c_cell_type_accuracy.tsv', 'figure_data/figure6_d_merfish_expression.tsv']:
    if not (CASE_OUTPUT / relative).is_file():
        raise FileNotFoundError(CASE_OUTPUT / relative)


## Step 3: Visualize the biological analysis

MERFISH cell-type accuracy asks whether the selected genes retain cellular identity in the assay that will be measured. Mean MERFISH expression asks whether the panel is supported by detectable biology. The paired comparison tests whether spatial references add biological information beyond the source transcriptome.

Only the manuscript panels are rendered below. Intermediate tables remain in the output directory for reproducibility but are not printed in this notebook. The quick hosted run uses only the methods/repeats executed above; use the full command to regenerate the complete multi-method comparison.

In [ ]:
figure_dir = CASE_OUTPUT / "figures"
plot_command = [sys.executable, str(ROOT / 'reproducibility/workflows/agent/plot_figure6.py'), '--accuracy', str(CASE_OUTPUT / 'figure_data/figure6_c_cell_type_accuracy.tsv'), '--expression', str(CASE_OUTPUT / 'figure_data/figure6_d_merfish_expression.tsv'), "--output-dir", str(figure_dir)]
subprocess.run(plot_command, cwd=ROOT, check=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for heading, relative, width in [('Figure 6c - MERFISH cell-type accuracy', 'figures/figure6_c.png', 430), ('Figure 6d - MERFISH expression support', 'figures/figure6_d.png', 430)]:
    display(Markdown(f"### {heading}"))
    display(Image(filename=str(CASE_OUTPUT / relative), width=width))


## Full manuscript command

Append the following paper-scale arguments to the workflow command:

```text
--panel-sizes 32,64,128 --training-seeds 1,2,3,4,5 --epochs 200 (omit --reference to use all five manifest-listed defaults)
```

This hosted run uses two real spatial references and two training seeds. The manuscript Figure 6c-d command uses five retrieved liver references and five training seeds. Figure 6e-j requires external probe-design backends and the validation-guided HPO run; this notebook does not fabricate those panels.